In [ ]:
import os
from openai import OpenAI
import gradio as gr
import json
from dotenv import load_dotenv
import base64
from io import BytesIO
from PIL import  Image

c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
load_dotenv()
model = "gpt-4.1-mini"
openai = OpenAI()

In [14]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [15]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"] }for h in history]
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
    )
    return response.choices[0].message.content

In [6]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [17]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"tool calling for the city {destination_city}")
    price = ticket_prices.get(destination_city.lower(),"Unknown City Price")
    return f"The price of a ticket to {destination_city} is {price}"



In [18]:
get_ticket_price("london")

tool calling for the city london


'The price of a ticket to london is $799'

In [38]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name":"get_ticket_price",
    "description":"Get the price of the return ticket to destination city.",
    "parameters": {
        "type":"object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description":"the particular city  customer wants to travel"
            }
        }
    }
}

In [39]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]


In [69]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
        return response
    return None

In [1]:




def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )
    if response.choices[0].finish_reason=="tool_calls":
        # handle tool calls
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=model, messages=messages)
        
    return response.choices[0].message

In [74]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


In [1]:
tools = [
    {
        "type":"function",
        "name":"get_horoscope",
        "description":"this is a function to get horoscope of the user",
        "parameters ":{
            "type":"object",
            "properties":{
                "sign":{
                    "type":"string",
                    "description":"this is the zodiac sign of the user."
                }
            },
            "required": ["sign"],
        },

    }
]

In [ ]:
import requests

url = "https://astropredict-daily-horoscopes-lucky-insights.p.rapidapi.com/horoscope"



headers = {
	"x-rapidapi-key": "fad3cf5020msh4873e60572eb5b6p18be83jsn8a2c1d80ad54",
	"x-rapidapi-host": "astropredict-daily-horoscopes-lucky-insights.p.rapidapi.com",
	"Content-Type": "application/json"
}



# print(response.json())


In [7]:
def get_horoscope(sign,type):
    querystring = {"lang":"en","zodiac":sign,"type":type,"timezone":"UTC"}
    response = requests.get(url, headers=headers, params=querystring)
    return response.json()

In [ ]:
get_horoscope("Aquarius","daily")

In [62]:
gr.ChatInterface(fn=chat).launch()

c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\utils.py:1177: UserWarning: Expected 1 arguments for function <function chat at 0x0000021CFD9A8EA0>, received 2.
  warnings.warn(
c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\utils.py:1185: UserWarning: Expected maximum 1 arguments for function <function chat at 0x0000021CFD9A8EA0>, received 2.
  warnings.warn(
c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\utils.py:1177: UserWarning: Expected 1 arguments for function <function chat at 0x0000021CFD9A9DA0>, received 2.
  warnings.warn(
c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\utils.py:1185: UserWarning: Expected maximum 1 arguments for function <function chat at 0x0000021CFD9A9DA0>, received 2.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\gradio\utils.py", line 1007, in async_wrapper
    response 

APIConnectionError: Connection error.